In [ ]:
import torch

print("CUDA available? ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))


In [ ]:
import pandas as pd
import glob
import os

In [ ]:
# Base data path (fixed for your project)
data_path = r"./data/raw"

# The merged sample file that uses only common columns
sample_file = os.path.join(data_path, "sample_merged_common_cols.csv")

In [ ]:
files = glob.glob(os.path.join(data_path, "*.csv"))
print("Total number of CSV files:", len(files))
for i, f in enumerate(files, 1):
    print(f"{i}.", os.path.basename(f))

총 CSV 파일 수: 10
1. 02-14-2018.csv
2. 02-15-2018.csv
3. 02-16-2018.csv
4. 02-20-2018.csv
5. 02-21-2018.csv
6. 02-22-2018.csv
7. 02-23-2018.csv
8. 02-28-2018.csv
9. 03-01-2018.csv
10. 03-02-2018.csv


In [ ]:
col_info = {}

for f in files:
    df_head = pd.read_csv(f, nrows=50)
    col_info[os.path.basename(f)] = list(df_head.columns)

print("=== Number of columns per file ===")
for name, cols in col_info.items():
    print(f"{name}: {len(cols)} columns")

# Common columns
common_cols = set.intersection(*[set(cols) for cols in col_info.values()])
print("\nNumber of common columns:", len(common_cols))

=== Number of columns per file ===
02-14-2018.csv: 80 columns
02-15-2018.csv: 80 columns
02-16-2018.csv: 80 columns
02-20-2018.csv: 84 columns
02-21-2018.csv: 80 columns
02-22-2018.csv: 80 columns
02-23-2018.csv: 80 columns
02-28-2018.csv: 80 columns
03-01-2018.csv: 80 columns
03-02-2018.csv: 80 columns

Number of common columns: 80


" 02-20-2018.csv " has momre columns than the others

In [7]:
# Find base columns (columns that exist in all files)
all_cols = list(col_info.values())
common_cols = set.intersection(*[set(c) for c in all_cols])
common_cols = list(common_cols)

# Columns of the specific file "02-20-2018.csv"
cols_0220 = col_info["02-20-2018.csv"]

extra_cols_0220 = [c for c in cols_0220 if c not in common_cols]

print("Number of common columns:", len(common_cols))
print("Number of columns in 02-20-2018.csv:", len(cols_0220))
print("Extra columns only in 02-20-2018.csv:", extra_cols_0220)


Number of common columns: 80
Number of columns in 02-20-2018.csv: 84
Extra columns only in 02-20-2018.csv: ['Flow ID', 'Src IP', 'Src Port', 'Dst IP']


This feature will be excluded for now. It can be analyzed separately for this specific day or incorporated in future extensions

In [ ]:
files = glob.glob(os.path.join(data_path, "*.csv"))

# Assume you already computed `common_cols` as above
base_cols = common_cols  # columns shared by all files

samples = []
sample_per_file = 3000  # rows per file

for f in files:
    try:
        # Read only the common columns
        df_part = pd.read_csv(f, nrows=sample_per_file, usecols=base_cols)
        samples.append(df_part)
        print(f"Sample extracted from: {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to read {os.path.basename(f)} → {e}")

df_sample = pd.concat(samples, ignore_index=True)
print("Merged sample shape:", df_sample.shape)

sample_path = os.path.join(data_path, "sample_merged_common_cols.csv")
df_sample.to_csv(sample_path, index=False)
print("Saved merged sample to:", sample_path)


ignore 02-20-2018.csv's 4 additional columns, use only columns common to other files => sample_merged_common_cols.csv

Only the 02-20-2018 capture provides four additional flow-level features ['Flow ID', 'Src IP', 'Src Port', 'Dst IP'].
In the current version, the main rule-based detector uses only the common feature set shared across all days. These extra features will be considered in future work
for day-specific or scenario-specific rules.

In [9]:
# Load all of column names
df_sample_head = pd.read_csv(sample_file, nrows=100)

print("Number of columns in merged sample:", len(df_sample_head.columns))
print("\n=== Column names in sample_merged_common_cols.csv ===")
for i, col in enumerate(df_sample_head.columns, 1):
    print(f"{i:2d}. {col}")

Number of columns in merged sample: 80

=== Column names in sample_merged_common_cols.csv ===
 1. Dst Port
 2. Protocol
 3. Timestamp
 4. Flow Duration
 5. Tot Fwd Pkts
 6. Tot Bwd Pkts
 7. TotLen Fwd Pkts
 8. TotLen Bwd Pkts
 9. Fwd Pkt Len Max
10. Fwd Pkt Len Min
11. Fwd Pkt Len Mean
12. Fwd Pkt Len Std
13. Bwd Pkt Len Max
14. Bwd Pkt Len Min
15. Bwd Pkt Len Mean
16. Bwd Pkt Len Std
17. Flow Byts/s
18. Flow Pkts/s
19. Flow IAT Mean
20. Flow IAT Std
21. Flow IAT Max
22. Flow IAT Min
23. Fwd IAT Tot
24. Fwd IAT Mean
25. Fwd IAT Std
26. Fwd IAT Max
27. Fwd IAT Min
28. Bwd IAT Tot
29. Bwd IAT Mean
30. Bwd IAT Std
31. Bwd IAT Max
32. Bwd IAT Min
33. Fwd PSH Flags
34. Bwd PSH Flags
35. Fwd URG Flags
36. Bwd URG Flags
37. Fwd Header Len
38. Bwd Header Len
39. Fwd Pkts/s
40. Bwd Pkts/s
41. Pkt Len Min
42. Pkt Len Max
43. Pkt Len Mean
44. Pkt Len Std
45. Pkt Len Var
46. FIN Flag Cnt
47. SYN Flag Cnt
48. RST Flag Cnt
49. PSH Flag Cnt
50. ACK Flag Cnt
51. URG Flag Cnt
52. CWE Flag Count
53. ECE

Label == attack type

In [ ]:
# Load the merged sample
df_sample = pd.read_csv(sample_file)

# Check basic info
print("Shape of df_sample:", df_sample.shape)
print("Columns include 'Label'? ->", "Label" in df_sample.columns)

# Print all unique Label values
print("\n=== Unique values in Label column ===")
unique_labels = df_sample["Label"].unique()
for i, v in enumerate(unique_labels, 1):
    print(f"{i:2d}. {v}")

# If you also want counts for each label
print("\n=== Label value counts (sorted) ===")
label_counts = df_sample["Label"].value_counts()
print(label_counts)

# If you want relative frequencies as well
print("\n=== Label value ratios (in %) ===")
label_ratios = (label_counts / len(df_sample) * 100).round(2)
print(label_ratios)


Attack type(56.3%): 
 1. FTP-BruteForce
 2. DoS attacks-GoldenEye
 3. DoS attacks-SlowHTTPTest
 4. DDoS attacks-LOIC-HTTP
 5. DDOS attack-LOIC-UDP
 6. Brute Force -Web
 7. Brute Force -XSS
 8. SQL Injection
 9. Bot

remove invalid Label row
row value literally "Label" -> invalid class label => delete

Attack type(56.3%): 
 1. FTP-BruteForce
 2. DoS attacks-GoldenEye
 3. DoS attacks-SlowHTTPTest
 4. DDoS attacks-LOIC-HTTP
 5. DDOS attack-LOIC-UDP
 6. Brute Force -Web
 7. Brute Force -XSS
 8. SQL Injection
 9. Bot

remove invalid Label row
row value literally "Label" -> invalid class label => delete

In [ ]:
data_path = r"./data"
sample_file = os.path.join(data_path, "sample_merged_common_cols.csv")

# Load original file
df = pd.read_csv(sample_file)

# Remove invalid row ("Label" as a value)
df_clean = df[df["Label"] != "Label"].reset_index(drop=True)

# Save cleaned version
clean_file = os.path.join(data_path, "sample_cleaned.csv")
df_clean.to_csv(clean_file, index=False)

print("Saved cleaned file to:", clean_file)
print("Old shape:", df.shape)
print("New shape:", df_clean.shape)
